In [56]:
import geopandas as gpd
import pandas as pd
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))  # adjust if notebook isn't 1 level deep

In [57]:
from viz.geo import show_cluster, show_selected_groups

In [58]:
gdf = gpd.read_file("../data/estaciones_agrupadas.geojson")
gdf.head(2)

,fecha,Decision,value,unit,fuente,nombre_programa,departamento,valor_original,limite_deteccion,limite_cuantificacion,valor_transformado,chla,grupo,geometry
0,2017-01-02,NaN,NaN,NaN,OAN,Agua Montes del Plata Empresa,COLONIA,8.900,0.100,0.1,8.900,8.900,68,POINT (401164.986 6214585.025)
1,2017-01-02,si,0.0068,mg/l,GEMS,NaN,NaN,NaN,NaN,NaN,NaN,6.800000090152,32,POINT (678380.002 6147956.963)


In [59]:
gdf = gdf.drop(columns=["nombre_programa", "departamento"])

In [60]:
gdf.groupby("grupo").agg(
    mediciones=("chla", "count")
).sort_values("mediciones", ascending=False)

,mediciones
grupo,
45,1020
33,612
32,432
34,292
44,291
...,...
3,3
55,3
97,3


In [61]:
gdf_agg = gdf.copy()
gdf_agg["grupo_final"] = gdf_agg["grupo"]

In [62]:
# UPM2
gdf_agg.loc[gdf_agg["grupo"].isin([45, 44, 43]), "grupo_final"] = 45

In [63]:
# juntamos [33, 32, 34] en el mismo grupo
gdf_agg.loc[gdf_agg["grupo"].isin([33, 32, 34]), "grupo_final"] = 33

In [64]:
# juntamos 51 y 53 en el mismo grupo
gdf_agg.loc[gdf_agg["grupo"].isin([51, 53]), "grupo_final"] = 51

In [65]:
gdf_agg.loc[gdf_agg["grupo"].isin([68, 65]), "grupo_final"] = 68

In [66]:
#gdf_agg.loc[gdf_agg["grupo"].isin([44]), "grupo_final"] = 44

In [67]:
gdf_agg.groupby("grupo_final").agg(
    mediciones=("chla", "count")
).sort_values("mediciones", ascending=False).head(10)

,mediciones
grupo_final,
45,1537
33,1336
51,393
68,370
42,156
18,109
26,102
24,91
56,89


In [68]:
show_cluster(43, gdf_agg, cluster_key="grupo_final")

/home/enzotrindade/ProyectoAA/viz/geo.py:12: UserWarning: The GeoSeries you are attempting to plot is composed of empty geometries. Nothing has been displayed.
  m = gdf[gdf[cluster_key] == num].to_crs(4326).explore(


In [69]:
selected_groups = [
    33,
    45,
    51,
    #42,
    68,
    18,
    26
]

In [70]:
show_selected_groups(gdf_agg, selected_groups, cluster_key="grupo_final")

In [71]:
#gdf_agg.rename
gdf_agg["grupo_nombre"] = None

In [72]:
def set_grupo_name(gdf, grupo_num, grupo_name):
    gdf.loc[gdf["grupo_final"] == grupo_num, "grupo_nombre"] = grupo_name

In [73]:
set_grupo_name(gdf_agg, 33, "LDS")
set_grupo_name(gdf_agg, 45, "RN-UPM2")
set_grupo_name(gdf_agg, 51, "RN-UPM1")
set_grupo_name(gdf_agg, 68, "RDP-MONTES")

In [74]:
gdf_agg.loc[~gdf_agg["grupo_final"].isin([33, 45, 51, 58]), "grupo_nombre"] = "NO-SELECCIONADO"

In [75]:
gdf_agg.head()

,fecha,Decision,value,unit,fuente,valor_original,limite_deteccion,limite_cuantificacion,valor_transformado,chla,grupo,geometry,grupo_final,grupo_nombre
0,2017-01-02,NaN,NaN,NaN,OAN,8.900,0.100,0.1,8.900,8.900,68,POINT (401164.986 6214585.025),68,NO-SELECCIONADO
1,2017-01-02,si,0.0068,mg/l,GEMS,NaN,NaN,NaN,NaN,6.800000090152,32,POINT (678380.002 6147956.963),33,LDS
2,2017-01-02,si,0.0032,mg/l,GEMS,NaN,NaN,NaN,NaN,3.1999999191612,33,POINT (679045.962 6144090.994),33,LDS
3,2017-01-02,NaN,NaN,NaN,OAN,4.400,0.100,0.1,4.400,4.400,68,POINT (402233.023 6210401.038),68,NO-SELECCIONADO
4,2017-01-02,NaN,NaN,NaN,OAN,5.900,0.100,0.1,5.900,5.900,68,POINT (401555.004 6213022.975),68,NO-SELECCIONADO


### Total mediciones de chl-a entre ubicaciones seleccionadas

Este es nuestro principal se conforma de 4 ubicaciones.

1. En primera instancia se aplico un algoritmo de clustering jerarquico para tener una agrupacion inicial de estos puntos
2. Luego se visualizaron estas agrupaciones y se juntaron clusters mas cercana formando los top-4 grupos finales

Este dataset es nuestro principal dataset y casi seguro sea nuestro train set (Podemos hacer mas revisiones.)

In [76]:
gdf_agg[gdf_agg["grupo_nombre"].isin(["LDS", "RN-UPM2", "RN-UPM1", "RDP-MONTES"])].shape


(3266, 14)

### Datos 'No Seleccionados'

No son datos descartados pero no forman parte de nuestro dataset principal

Evaluaremos estos puntos en otro apartado para seleccionar sets de test y validacion.

In [77]:
gdf_agg[gdf_agg["grupo_nombre"] == "NO-SELECCIONADO"].shape

(3215, 14)

In [84]:
(gdf_agg[gdf_agg["grupo_nombre"].isin(["LDS", "RN-UPM2", "RN-UPM1", "RDP-MONTES"])]["Decision"] == "no").sum()

np.int64(29)

### De los seleccionados cuantos no tienen un valor de deteccion?

In [91]:
seleccionados = gdf_agg[gdf_agg["grupo_nombre"].isin(["LDS", "RN-UPM2", "RN-UPM1", "RDP-MONTES"])]
mask_non_numeric = pd.to_numeric(seleccionados['chla'], errors='coerce').isna()
chla_non_num = seleccionados[mask_non_numeric]
LD_LC_MASK = (chla_non_num == "<LD") | (chla_non_num == '<LC') | (chla_non_num == 'LD<X<LC') | (chla_non_num == 'LD<x<LC')
LD_LC_MASK["chla"].shape[0]

689